In [ ]:
"""
Betel Palm (Areca catechu) Planting Advisor — ML Training Pipeline
===================================================================
This script trains THREE separate Random Forest classifiers to help
farmers make data-driven decisions when planting betel palm saplings.

  Model 1 — season_label   : What is the best planting season?
  Model 2 — drought_label  : How high is the drought / drying risk?
  Model 3 — crop_label     : Which companion crop gives the best shade?

How to run:
    python betel_palm_ml.py

Output files produced:
    betel_palm_ml_report.png  →  evaluation charts (confusion matrix,
                                  feature importance, class distribution,
                                  cross-validation accuracy)
    models/season_label_rf.joblib   →  saved Season model
    models/drought_label_rf.joblib  →  saved Drought model
    models/crop_label_rf.joblib     →  saved Crop model
"""

# ─────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────

import numpy as np          # numerical operations, array handling
import pandas as pd         # tabular data (DataFrame)
import matplotlib           # plotting library
matplotlib.use("Agg")       # use non-interactive backend (saves to file, no popup)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec   # advanced subplot layout
import warnings, os, joblib              # utilities: suppress warnings, file paths, model saving
warnings.filterwarnings("ignore")        # hide sklearn version/convergence warnings

# sklearn — machine learning toolkit
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# RandomForestClassifier : builds many decision trees and votes on the result
# GradientBoostingClassifier : imported but RandomForest is used (kept for future experiments)

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
# train_test_split : splits data into training set and test set
# cross_val_score  : evaluates model on multiple folds to check consistency
# StratifiedKFold  : ensures each fold has the same class ratio as the full dataset

from sklearn.preprocessing import LabelEncoder
# LabelEncoder : converts text labels (e.g. "banana") to integers — imported for reference

from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score)
# classification_report  : shows precision, recall, f1-score per class
# confusion_matrix       : shows which classes get confused with each other
# ConfusionMatrixDisplay : renders the matrix as a heatmap
# accuracy_score         : overall % of correct predictions

from sklearn.pipeline import Pipeline              # not used directly, available for extensions
from sklearn.inspection import permutation_importance  # available for deeper feature analysis

# Fix random seed so results are reproducible every run
np.random.seed(42)


# ─────────────────────────────────────────────────────────────────────
# SECTION 1 — LABEL ENCODING MAPS
# ─────────────────────────────────────────────────────────────────────
# ML models only understand numbers, not strings.
# These dictionaries convert human-readable inputs to integer codes.
# The same maps are used for both dataset generation AND prediction.

# Climate zone of the farm
ZONE_MAP = {
    "tropical":    0,   # hot & humid year-round
    "subtropical": 1,   # mild winters, warm summers
    "semiarid":    2,   # low rainfall, dry periods
    "coastal":     3,   # near sea, moderate temperature
}

# Soil type of the farm
SOIL_MAP = {
    "loamy":    0,   # well-drained, ideal for betel palm
    "clay":     1,   # water-retaining, risk of waterlogging
    "sandy":    2,   # fast-draining, loses moisture quickly
    "laterite": 3,   # acidic, mineral-rich (common in South India)
    "alluvial": 4,   # river-bank soil, very fertile
}

# Irrigation availability
IRR_MAP = {
    "yes":     0,   # drip or channel irrigation available
    "partial": 1,   # seasonal / limited irrigation
    "no":      2,   # fully rain-fed farm
}

# Geographic region (affects crop suitability and monsoon timing)
REGION_MAP = {
    "india_south": 0,   # South India / Sri Lanka
    "india_ne":    1,   # Northeast India / Bangladesh
    "sea":         2,   # Southeast Asia (Thailand, Myanmar)
    "pacific":     3,   # Pacific Islands
    "other":       4,   # any other tropical region
}

# ─────────────────────────────────────────────────────────────────────
# OUTPUT CLASS LABELS
# ─────────────────────────────────────────────────────────────────────
# Human-readable names for each class the models predict.
# The index in the list matches the integer the model outputs internally.

SEASON_LABELS = [
    "Early monsoon (May-Aug)",       # class 0 — tropical zones
    "Summer monsoon (Jun-Sep)",      # class 1 — subtropical zones
    "Post-monsoon (Jul-Oct)",        # class 2 — semiarid zones
    "Pre-monsoon coastal (Apr-Jul)", # class 3 — coastal zones
]

DROUGHT_LABELS = [
    "Low risk",    # class 0 — good conditions, plant safely
    "Medium risk", # class 1 — some precautions needed
    "High risk",   # class 2 — irrigation / mulching mandatory
]

CROP_LABELS = [
    "Banana",          # class 0 — fast-growing, tropical / subtropical
    "Coconut",         # class 1 — long-term shade, coastal / tropical
    "Tapioca/Cassava", # class 2 — drought-tolerant, semiarid
    "Black pepper",    # class 3 — climbs betel palm, dual income
    "Turmeric",        # class 4 — ground-level shade, loamy soil
    "Jackfruit",       # class 5 — dense canopy, deep roots
]

# Feature names — must match the order of columns fed into the model
FEATURES = [
    "zone",          # encoded climate zone (0-3)
    "soil",          # encoded soil type (0-4)
    "rainfall_mm",   # annual rainfall in millimetres
    "temp_c",        # average temperature in degrees C
    "irrigation",    # encoded irrigation level (0-2)
    "region",        # encoded geographic region (0-4)
    "humidity_pct",  # relative humidity percentage
    "elevation_m",   # farm elevation in metres above sea level
]


# ─────────────────────────────────────────────────────────────────────
# SECTION 2 — DOMAIN RULE ENGINES (Label Generators)
# ─────────────────────────────────────────────────────────────────────
# Since we don't have real field data, we simulate ground-truth labels
# using expert agronomic rules. The ML models then LEARN these patterns
# from the generated dataset — plus some added noise (see generate_dataset).

def _season_label(zone, rainfall, irrigation, temp):
    """
    Determine the best planting season based on climate zone.

    Logic:
      - Tropical zones  : plant at Early monsoon onset (May-Aug)
                          soil is moist, temperature warm, roots establish fast
      - Subtropical     : Summer monsoon (Jun-Sep)
                          wait for monsoon, night temps must stay above 22 degrees C
      - Semiarid        : Post-monsoon (Jul-Oct)
                          plant only AFTER monsoon arrives, soil needs 30%+ moisture
      - Coastal         : Pre-monsoon coastal (Apr-Jul)
                          sea breezes moderate temperature, plant before peak heat

    Parameters
    ----------
    zone       : int — encoded climate zone (0=tropical to 3=coastal)
    rainfall   : int — annual rainfall in mm
    irrigation : int — irrigation level
    temp       : int — average temperature in degrees C

    Returns
    -------
    int — class index (0-3) matching SEASON_LABELS
    """
    if zone == 0:    # tropical
        return 0     # Early monsoon
    elif zone == 1:  # subtropical
        return 1     # Summer monsoon
    elif zone == 2:  # semiarid
        return 2     # Post-monsoon
    else:            # coastal
        return 3     # Pre-monsoon coastal


def _drought_label(zone, soil, rainfall, irrigation, temp, humidity):
    """
    Estimate drought / drying risk for a sapling based on farm conditions.

    Uses a points-based scoring system:
      Each risk factor ADDS points, each protective factor SUBTRACTS points.
      Final score maps to Low / Medium / High risk.

    Risk factors (add points):
      +2  sandy soil          — loses moisture very quickly
      +2  no irrigation       — fully dependent on rain
      +2  semiarid zone       — structurally low rainfall region
      +2  rainfall < 800mm    — critically low annual water
      +1  rainfall < 1200mm   — moderately low water
      +1  temp > 35 degrees C — heat stress accelerates evaporation
      +1  humidity < 50%      — dry air pulls moisture from soil

    Protective factors (subtract points):
      -1  loamy or alluvial soil — holds moisture better
      -1  irrigation available   — can supplement rain

    Thresholds:
      score 0-2  : Low risk    (class 0)
      score 3-5  : Medium risk (class 1)
      score 6+   : High risk   (class 2)

    Parameters
    ----------
    zone, soil, rainfall, irrigation, temp, humidity : int

    Returns
    -------
    int — class index (0=Low, 1=Medium, 2=High)
    """
    score = 0

    # --- Risk factors (each condition that increases drought risk) ---
    if soil == 2:          score += 2   # sandy soil drains too fast
    if irrigation == 2:    score += 2   # no irrigation, rain-fed only
    if zone == 2:          score += 2   # semiarid zone, structurally dry
    if rainfall < 800:     score += 2   # critically low annual rainfall
    elif rainfall < 1200:  score += 1   # moderately low rainfall
    if temp > 35:          score += 1   # high temp causes more evaporation
    if humidity < 50:      score += 1   # dry air accelerates moisture loss

    # --- Protective factors (conditions that reduce drought risk) ---
    if soil in (0, 4):     score -= 1   # loamy or alluvial soil holds moisture well
    if irrigation == 0:    score -= 1   # has irrigation available to top up water

    # Score cannot go below 0
    score = max(score, 0)

    # Map total score to a risk class
    if score <= 2:   return 0   # Low risk
    elif score <= 5: return 1   # Medium risk
    else:            return 2   # High risk


def _crop_label(zone, soil, region, temp, rainfall, irrigation):
    """
    Determine the single best companion / shade crop for betel palm saplings.

    Uses a points-based scoring system across 6 candidate crops.
    Each crop gains points when the farm conditions match its preferences.
    The crop with the highest total score is returned.

    Scoring rules:
      Banana       : +3 if tropical/subtropical AND rainfall > 1200mm
                     +1 if irrigation available
                     -2 if rainfall < 900mm (cannot survive very dry conditions)
      Coconut      : +3 if tropical or coastal zone
      Tapioca      : +4 if semiarid zone (most drought-tolerant option)
                     +2 if rainfall < 900mm
      Black pepper : +3 if South India, NE India or SE Asia region
                        (traditionally intercropped alongside betel palm)
      Turmeric     : +2 if loamy or alluvial soil (needs good drainage + nutrients)
      Jackfruit    : +2 if tropical or subtropical zone (dense permanent canopy)

    Parameters
    ----------
    zone, soil, region, temp, rainfall, irrigation : int

    Returns
    -------
    int — class index matching CROP_LABELS
    """
    # Initialise score table — all crops start at zero
    scores = {
        "Banana":          0,
        "Coconut":         0,
        "Tapioca/Cassava": 0,
        "Black pepper":    0,
        "Turmeric":        0,
        "Jackfruit":       0,
    }

    # Banana thrives in tropical/subtropical zones with good rainfall
    if zone in (0, 1) and rainfall > 1200:
        scores["Banana"] += 3

    # Coconut is a natural long-term shade partner in tropical/coastal zones
    if zone in (0, 3):
        scores["Coconut"] += 3

    # Tapioca/Cassava is the most drought-tolerant — best for semiarid zones
    if zone == 2:
        scores["Tapioca/Cassava"] += 4

    # Black pepper traditionally climbs betel palms — strong regional match
    if region in (0, 1, 2):   # india_south, india_ne, sea
        scores["Black pepper"] += 3

    # Turmeric grows at ground level — works well in fertile, well-drained soil
    if soil in (0, 4):         # loamy or alluvial
        scores["Turmeric"] += 2

    # Jackfruit provides dense permanent canopy in warm humid zones
    if zone in (0, 1):         # tropical or subtropical
        scores["Jackfruit"] += 2

    # Adjust for low rainfall — banana struggles below 900mm, tapioca dominates
    if rainfall < 900:
        scores["Tapioca/Cassava"] += 2
        scores["Banana"] -= 2

    # Irrigation availability makes banana more viable (it needs consistent water)
    if irrigation == 0:        # irrigation available
        scores["Banana"] += 1

    # Return the index of the highest-scoring crop in CROP_LABELS
    return CROP_LABELS.index(max(scores, key=scores.get))


# ─────────────────────────────────────────────────────────────────────
# SECTION 3 — SYNTHETIC DATASET GENERATION
# ─────────────────────────────────────────────────────────────────────

def generate_dataset(n=3000):
    """
    Generate a synthetic labelled dataset of n farm profiles.

    Since real field survey data is not available, we simulate realistic
    farm conditions using random sampling with probability weights that
    reflect real-world distribution (e.g. more tropical than semiarid farms).

    Realistic correlations are applied after sampling:
      - Tropical zones get boosted rainfall (+25%) and humidity (+10 pts)
      - Semiarid zones get reduced rainfall (-35%) and humidity (-15 pts)

    A 7% random noise is injected into season and drought labels to
    prevent the model from learning a perfect deterministic rule.
    Real-world data always contains edge cases and exceptions.

    Parameters
    ----------
    n : int — number of synthetic farm samples to generate (default 3000)

    Returns
    -------
    pd.DataFrame — shape (n, 11) with 8 feature columns + 3 label columns
    """
    rows = []   # will hold one list per farm sample

    for _ in range(n):

        # ── Sample input features ────────────────────────────────────

        # Climate zone — tropical is most common globally for betel palm farming
        # p = probability weights (must sum to 1.0)
        zone = np.random.choice(
            list(ZONE_MAP.values()),
            p=[0.35, 0.25, 0.20, 0.20]   # tropical most common
        )

        # Soil type — loamy is the most common agricultural soil worldwide
        soil = np.random.choice(
            list(SOIL_MAP.values()),
            p=[0.30, 0.20, 0.20, 0.15, 0.15]
        )

        # Annual rainfall — normal distribution centred at 1500mm
        # np.clip() ensures the value stays within the valid range 400-3600mm
        rainfall = int(np.clip(np.random.normal(1500, 600), 400, 3600))

        # Average temperature — centred at 28 degrees C (ideal for betel palm)
        temp = int(np.clip(np.random.normal(28, 4), 19, 41))

        # Irrigation: 40% have full irrigation, 30% partial, 30% rain-fed
        irrigation = np.random.choice([0, 1, 2], p=[0.40, 0.30, 0.30])

        # Region — South India and SE Asia are the primary betel palm growing regions
        region = np.random.choice(
            list(REGION_MAP.values()),
            p=[0.30, 0.20, 0.25, 0.10, 0.15]
        )

        # Relative humidity — centred at 65% for tropical average
        humidity = int(np.clip(np.random.normal(65, 15), 30, 98))

        # Farm elevation — most betel palms are grown in lowlands under 200m
        elevation = int(np.clip(np.random.normal(120, 80), 0, 600))

        # ── Apply realistic climate correlations ─────────────────────
        # Tropical zones naturally have more rainfall and higher humidity
        if zone == 0:   # tropical
            rainfall = int(min(rainfall * 1.25, 3600))  # boost rainfall by 25%
            humidity = min(humidity + 10, 98)            # boost humidity by 10 pts

        # Semiarid zones have structurally lower rainfall and drier air
        elif zone == 2:   # semiarid
            rainfall = max(int(rainfall * 0.65), 400)   # reduce rainfall by 35%
            humidity = max(humidity - 15, 30)            # reduce humidity by 15 pts

        # ── Generate target labels from domain rules ─────────────────
        # These call the rule-engine functions defined in Section 2
        season  = _season_label(zone, rainfall, irrigation, temp)
        drought = _drought_label(zone, soil, rainfall, irrigation, temp, humidity)
        crop    = _crop_label(zone, soil, region, temp, rainfall, irrigation)

        # ── Inject random label noise (7% probability) ───────────────
        # This simulates real-world variability and edge cases.
        # Without noise the model would achieve close to 100% accuracy
        # by simply memorising the rules — which is not useful in practice.
        if np.random.random() < 0.07:
            season = np.random.randint(0, 4)   # randomly override season label
        if np.random.random() < 0.07:
            drought = np.random.randint(0, 3)  # randomly override drought label

        # Append all 11 values (8 features + 3 labels) as one row
        rows.append([
            zone, soil, rainfall, temp, irrigation, region,
            humidity, elevation,
            season, drought, crop
        ])

    # Convert list of lists into a structured pandas DataFrame
    df = pd.DataFrame(
        rows,
        columns=FEATURES + ["season_label", "drought_label", "crop_label"]
    )
    return df


# ─────────────────────────────────────────────────────────────────────
# SECTION 4 — MODEL TRAINING
# ─────────────────────────────────────────────────────────────────────

def train_models(df):
    """
    Train three independent Random Forest classifiers (one per target).

    Why Random Forest?
      - Handles both numerical (rainfall, temp) and categorical (zone, soil) inputs
      - Robust to outliers and noisy labels
      - Provides built-in feature importance scores at no extra cost
      - No need to scale or normalise features (unlike SVM or neural nets)
      - class_weight='balanced' compensates for imbalanced classes automatically

    For each of the three models this function:
      1. Extracts the target column (y)
      2. Splits into 80% train / 20% test (stratified to keep class ratios intact)
      3. Trains a Random Forest on the training split
      4. Measures accuracy on the held-out test split
      5. Runs 5-fold cross-validation to confirm generalisation

    Parameters
    ----------
    df : pd.DataFrame — labelled dataset from generate_dataset()

    Returns
    -------
    dict — one key per model:
           {"model", "X_te", "y_te", "y_pred", "acc", "cv_mean", "cv_std", "labels"}
    """
    # Extract the 8 input feature columns as a 2D NumPy array
    # Shape: (3000, 8) — 3000 samples, 8 features each
    X = df[FEATURES].values

    results = {}   # stores trained model + evaluation metrics for each target

    # ── Loop over the three prediction targets ────────────────────────
    # Each tuple defines: (target column name, class label list, RF hyperparameters)
    for target, label_list, clf_kw in [

        # Season model — 4 classes, moderate depth
        ("season_label",  SEASON_LABELS,
         dict(n_estimators=200, max_depth=8, min_samples_leaf=4)),

        # Drought model — 3 classes, shallower depth to avoid overfitting
        # the rare "High risk" class (very few samples relative to "Low risk")
        ("drought_label", DROUGHT_LABELS,
         dict(n_estimators=200, max_depth=6, min_samples_leaf=4)),

        # Crop model — 6 classes, deeper trees for more nuanced crop interactions
        ("crop_label",    CROP_LABELS,
         dict(n_estimators=200, max_depth=10, min_samples_leaf=3)),
    ]:

        # Extract the target labels (y) for this specific model
        y = df[target].values  # shape: (3000,)

        # ── Train / Test Split ────────────────────────────────────────
        # test_size=0.20 : 80% of data for training, 20% reserved for testing
        # stratify=y     : ensures both splits contain the same proportion
        #                  of each class (important for imbalanced datasets)
        # random_state   : fixed for reproducibility
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.20, random_state=42, stratify=y
        )

        # ── Build and Train the Random Forest ────────────────────────
        # n_estimators=200     : 200 decision trees vote together
        # max_depth            : limits tree growth to prevent overfitting
        # min_samples_leaf     : each leaf must have at least N samples
        #                        (also prevents overfitting to rare samples)
        # class_weight=balanced: automatically up-weights rare classes
        #                        so "High risk" (rare) is not ignored vs "Low risk"
        # n_jobs=-1            : parallel training using all CPU cores
        # random_state=42      : reproducible tree splits
        model = RandomForestClassifier(
            random_state=42,
            n_jobs=-1,
            class_weight="balanced",
            **clf_kw   # unpack n_estimators, max_depth, min_samples_leaf
        )
        model.fit(X_tr, y_tr)   # train on 80% of the data

        # ── Evaluate on the Held-Out Test Set ─────────────────────────
        y_pred = model.predict(X_te)           # predict class for each test sample
        acc    = accuracy_score(y_te, y_pred)  # fraction of correct predictions

        # ── 5-Fold Cross-Validation ───────────────────────────────────
        # Splits the ENTIRE dataset into 5 equal folds.
        # Trains on 4 folds, tests on the remaining 1 fold.
        # Repeats 5 times so every sample is tested exactly once.
        # cv.mean() = average accuracy across all 5 folds
        # cv.std()  = how much accuracy varies between folds (lower = more stable)
        cv = cross_val_score(
            model, X, y,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
            scoring="accuracy",
            n_jobs=-1
        )

        # Store model + all metrics for chart generation and inference
        results[target] = {
            "model":   model,      # trained sklearn Random Forest object
            "X_te":    X_te,       # test set features (for confusion matrix)
            "y_te":    y_te,       # true test labels
            "y_pred":  y_pred,     # model's predicted labels on test set
            "acc":     acc,        # test set accuracy (0.0 to 1.0)
            "cv_mean": cv.mean(),  # mean 5-fold cross-validation accuracy
            "cv_std":  cv.std(),   # standard deviation across the 5 CV folds
            "labels":  label_list, # human-readable class names for reports
        }

        # ── Print Per-Class Metrics ───────────────────────────────────
        # precision : of everything predicted as class X, how many were truly X?
        # recall    : of all actual class X samples, what fraction did we catch?
        # f1-score  : combined score — harmonic mean of precision and recall
        # support   : total number of test samples per class
        print(f"\n{'─'*52}")
        print(f"  Model : {target}")
        print(f"  Test accuracy  : {acc*100:.1f}%")
        print(f"  CV accuracy    : {cv.mean()*100:.1f}% +/- {cv.std()*100:.1f}%")
        print(f"\n{classification_report(y_te, y_pred, target_names=label_list)}")

    return results


# ─────────────────────────────────────────────────────────────────────
# SECTION 5 — EVALUATION CHARTS
# ─────────────────────────────────────────────────────────────────────

def plot_results(results, df, out="betel_palm_ml_report.png"):
    """
    Generate a 4-row evaluation dashboard and save it as a PNG file.

    Layout (4 rows x 3 columns):
      Row 0 — Confusion matrices  : actual vs predicted class heatmaps
      Row 1 — Feature importance  : which inputs matter most per model
      Row 2 — Class distribution  : how many samples per class in training data
      Row 3 — CV accuracy bars    : compare all three models side by side

    Parameters
    ----------
    results : dict        — output from train_models()
    df      : pd.DataFrame — training dataset (for class distribution charts)
    out     : str         — file path for the saved PNG image

    Returns
    -------
    str — path to the saved chart file
    """
    # Create one large figure (width x height in inches)
    fig = plt.figure(figsize=(20, 22), facecolor="#F8F8F5")
    fig.suptitle("Betel Palm Planting Advisor — ML Model Report",
                 fontsize=20, fontweight="bold", y=0.99, color="#1a1a1a")

    # GridSpec divides the figure into a 4-row by 3-column grid
    # hspace/wspace control vertical and horizontal spacing between subplots
    gs = gridspec.GridSpec(4, 3, figure=fig,
                           hspace=0.55, wspace=0.38,
                           top=0.96, bottom=0.03)

    # One display colour per model for visual consistency across all rows
    COLORS       = ["#3B6D11", "#185FA5", "#854F0B"]  # green, blue, amber
    target_names = ["season_label", "drought_label", "crop_label"]
    titles       = ["Planting Season", "Drought Risk", "Shade Crop"]

    # ── Row 0: Confusion Matrices ─────────────────────────────────────
    # A confusion matrix grid shows:
    #   Rows    = actual (true) class
    #   Columns = predicted class
    #   Diagonal cells = correct predictions (want these to be large)
    #   Off-diagonal   = errors (e.g. predicted Low risk but actual was High risk)
    for col, (key, color) in enumerate(zip(target_names, COLORS)):
        r  = results[key]
        ax = fig.add_subplot(gs[0, col])

        # Compute the confusion matrix from true vs predicted label arrays
        cm = confusion_matrix(r["y_te"], r["y_pred"])

        # Render the matrix as a colour-coded heatmap — darker = more samples
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=r["labels"]
        )
        disp.plot(
            ax=ax, colorbar=False,
            cmap="Greens" if col == 0 else "Blues" if col == 1 else "Oranges"
        )
        ax.set_title(
            f"{titles[col]}\n"
            f"Test acc: {r['acc']*100:.1f}%  |  CV: {r['cv_mean']*100:.1f}%+/-{r['cv_std']*100:.1f}%",
            fontsize=9, pad=6, color="#222"
        )
        ax.tick_params(axis="x", labelsize=7, rotation=30)
        ax.tick_params(axis="y", labelsize=7)
        ax.set_xlabel("Predicted", fontsize=8)
        ax.set_ylabel("Actual", fontsize=8)
        for text in ax.texts:
            text.set_fontsize(8)

    # ── Row 1: Feature Importance ─────────────────────────────────────
    # Random Forest measures importance as Mean Decrease in Impurity (Gini importance).
    # A high score means that feature is used heavily to split decision nodes.
    # Long bar = that input (e.g. rainfall_mm) is critical for this prediction.
    # Short bar = that input (e.g. elevation_m) barely affects the prediction.
    for col, (key, color) in enumerate(zip(target_names, COLORS)):
        r   = results[key]
        ax  = fig.add_subplot(gs[1, col])
        imp = r["model"].feature_importances_   # array of 8 importance scores
        idx = np.argsort(imp)                   # sort indices from least to most important

        # Horizontal bar chart — longest bar = most important feature
        ax.barh([FEATURES[i] for i in idx], imp[idx], color=color, alpha=0.82)
        ax.set_title(f"Feature importance — {titles[col]}", fontsize=9)
        ax.tick_params(axis="both", labelsize=8)
        ax.set_xlabel("Mean decrease in impurity", fontsize=7)
        ax.spines[["top", "right"]].set_visible(False)

    # ── Row 2: Class Distribution ─────────────────────────────────────
    # Shows how many training samples belong to each output class.
    # Severely imbalanced classes (e.g. "High risk" = only 6%) can cause
    # the model to ignore minority classes — mitigated by class_weight="balanced".
    dist_data = [
        (df["season_label"].value_counts().sort_index(),  SEASON_LABELS,  COLORS[0]),
        (df["drought_label"].value_counts().sort_index(), DROUGHT_LABELS, COLORS[1]),
        (df["crop_label"].value_counts().sort_index(),    CROP_LABELS,    COLORS[2]),
    ]
    for col, (counts, labels, color) in enumerate(dist_data):
        ax = fig.add_subplot(gs[2, col])
        bars = ax.bar(range(len(labels)), counts.values, color=color, alpha=0.75)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=7)
        ax.set_title(f"Class distribution — {titles[col]}", fontsize=9)
        ax.set_ylabel("Samples", fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)
        # Label each bar with its exact sample count
        for bar in bars:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 10,
                str(int(bar.get_height())),
                ha="center", fontsize=7
            )

    # ── Row 3: Cross-Validation Accuracy Comparison ────────────────────
    # Compares all three models on the same chart.
    # Error bars show +/- std deviation across the 5 folds.
    # Small error bar = model is stable regardless of which data it sees.
    # Large error bar = model performance varies a lot — possible instability.
    ax_cv     = fig.add_subplot(gs[3, :])   # spans all 3 columns
    models_cv = [(t, results[t]["cv_mean"] * 100, results[t]["cv_std"] * 100)
                 for t in target_names]

    bars = ax_cv.bar(
        # Clean x-axis labels (remove "_label" suffix)
        [t[0].replace("_label", "").replace("_", " ").title() for t in models_cv],
        [t[1] for t in models_cv],      # bar height = mean CV accuracy
        yerr=[t[2] for t in models_cv], # error bars = +/- std deviation
        color=COLORS, alpha=0.82, capsize=6, width=0.45,
        error_kw=dict(elinewidth=1.2, ecolor="#555")
    )
    ax_cv.set_ylim(0, 110)
    ax_cv.set_ylabel("5-Fold CV Accuracy (%)", fontsize=10)
    ax_cv.set_title("Cross-Validation Accuracy — All Three Models", fontsize=11)
    ax_cv.spines[["top", "right"]].set_visible(False)

    # Reference line at 90% accuracy — our minimum quality target
    ax_cv.axhline(90, color="#aaa", lw=0.8, ls="--")
    ax_cv.text(2.6, 91, "90% target", fontsize=8, color="#888")

    # Annotate each bar with its mean accuracy value
    for bar, (_, mean, std) in zip(bars, models_cv):
        ax_cv.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + std + 1,
            f"{mean:.1f}%",
            ha="center", fontsize=10, fontweight="bold"
        )

    # Save the completed figure to disk as a PNG image
    plt.savefig(out, dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor())
    print(f"\n  Chart saved -> {out}")
    return out


# ─────────────────────────────────────────────────────────────────────
# SECTION 6 — PREDICTION FUNCTION
# ─────────────────────────────────────────────────────────────────────

def predict(models_dict, zone="tropical", soil="loamy",
            rainfall_mm=1500, temp_c=28, irrigation="yes",
            region="india_south", humidity_pct=70, elevation_m=100):
    """
    Use all three trained models to predict recommendations for a single farm.

    Steps inside this function:
      1. Convert text inputs (e.g. "tropical") to integers using encoding maps
      2. Reshape into a 1-row NumPy array of shape (1, 8)
      3. Run model.predict()       — returns the most likely class index
      4. Run model.predict_proba() — returns probability for EVERY class
      5. Return results as a nested dictionary

    Parameters
    ----------
    models_dict  : dict — output from train_models() containing trained models
    zone         : str  — "tropical" | "subtropical" | "semiarid" | "coastal"
    soil         : str  — "loamy" | "clay" | "sandy" | "laterite" | "alluvial"
    rainfall_mm  : int  — annual rainfall in mm (valid range 400-3600)
    temp_c       : int  — average temperature in degrees C (valid range 19-41)
    irrigation   : str  — "yes" | "partial" | "no"
    region       : str  — "india_south" | "india_ne" | "sea" | "pacific" | "other"
    humidity_pct : int  — relative humidity percentage (valid range 30-98)
    elevation_m  : int  — farm elevation in metres (valid range 0-600)

    Returns
    -------
    dict — nested structure:
      {
        "season_label":  {"prediction": str, "confidence": str, "probabilities": dict},
        "drought_label": {"prediction": str, "confidence": str, "probabilities": dict},
        "crop_label":    {"prediction": str, "confidence": str, "probabilities": dict},
      }

    Example call
    ------------
    result = predict(
        results,
        zone="tropical", soil="loamy", rainfall_mm=1800, temp_c=28,
        irrigation="yes", region="india_south", humidity_pct=75, elevation_m=80
    )
    print(result["season_label"]["prediction"])   # "Early monsoon (May-Aug)"
    print(result["drought_label"]["confidence"])  # "83.0%"
    print(result["crop_label"]["probabilities"])  # {"Banana": "97.2%", ...}
    """

    # ── Step 1: Encode all text inputs to integer codes ───────────────
    # The model was trained on numeric arrays, not raw strings.
    # We use the exact same encoding maps defined in Section 1 above.
    x = np.array([[
        ZONE_MAP[zone],          # e.g. "tropical" -> 0
        SOIL_MAP[soil],          # e.g. "loamy"    -> 0
        rainfall_mm,             # already numeric, passed as-is
        temp_c,                  # already numeric, passed as-is
        IRR_MAP[irrigation],     # e.g. "yes"      -> 0
        REGION_MAP[region],      # e.g. "india_south" -> 0
        humidity_pct,            # already numeric
        elevation_m,             # already numeric
    ]])
    # x.shape is (1, 8) — 1 sample row, 8 feature columns

    out = {}   # will hold prediction results for all three models

    # ── Step 2: Run each trained model ───────────────────────────────
    for key, labels in [
        ("season_label",  SEASON_LABELS),
        ("drought_label", DROUGHT_LABELS),
        ("crop_label",    CROP_LABELS),
    ]:
        m = models_dict[key]["model"]   # retrieve the trained Random Forest

        # model.predict(x) returns the most likely class as an integer
        # e.g. 0 -> "Early monsoon (May-Aug)" for the season model
        pred  = m.predict(x)[0]

        # model.predict_proba(x) returns a probability for every class
        # e.g. [0.767, 0.10, 0.08, 0.05] means 76.7% confident it's class 0
        proba = m.predict_proba(x)[0]

        out[key] = {
            # Convert integer class index back to a human-readable label
            "prediction":    labels[pred],

            # Confidence = the highest probability among all classes
            "confidence":    f"{proba.max() * 100:.1f}%",

            # Full breakdown: probability for every possible class
            # Useful for showing the farmer the second-best crop option
            "probabilities": {
                labels[i]: f"{p * 100:.1f}%"
                for i, p in enumerate(proba)
            },
        }

    return out


# ─────────────────────────────────────────────────────────────────────
# SECTION 7 — SAVE TRAINED MODELS TO DISK
# ─────────────────────────────────────────────────────────────────────

def save_models(results, path="models"):
    """
    Persist all three trained models to disk using joblib serialisation.

    Why joblib over pickle?
      - More efficient for large NumPy arrays (which Random Forests contain)
      - Faster save and load times for large models
      - The saved file can be loaded in any Python environment without
        re-running the full training pipeline

    Saved file names:
      models/season_label_rf.joblib
      models/drought_label_rf.joblib
      models/crop_label_rf.joblib

    How to reload a saved model later (in a separate script):
      import joblib, numpy as np
      model = joblib.load("models/season_label_rf.joblib")
      # Features must be in the same order as FEATURES list above:
      # [zone, soil, rainfall_mm, temp_c, irrigation, region, humidity_pct, elevation_m]
      prediction = model.predict([[0, 0, 1500, 28, 0, 0, 70, 100]])
      print(SEASON_LABELS[prediction[0]])

    Parameters
    ----------
    results : dict — output from train_models()
    path    : str  — directory to save models into (created if it doesn't exist)
    """
    # Create the output directory if it doesn't exist yet
    os.makedirs(path, exist_ok=True)

    for key in ("season_label", "drought_label", "crop_label"):
        # Build the full file path, e.g. "models/season_label_rf.joblib"
        fpath = os.path.join(path, f"{key}_rf.joblib")

        # Serialise (save) the trained model object to disk
        joblib.dump(results[key]["model"], fpath)

    print(f"\n  Models saved -> {path}/")


# ─────────────────────────────────────────────────────────────────────
# SECTION 8 — MAIN EXECUTION PIPELINE
# ─────────────────────────────────────────────────────────────────────

def main():
    """
    Orchestrates the full pipeline end-to-end in 4 steps:

      Step 1 — Generate dataset : create 3,000 synthetic farm profiles with labels
      Step 2 — Train models     : train and evaluate all three Random Forest classifiers
      Step 3 — Plot results     : generate and save evaluation charts as PNG
      Step 4 — Save models      : persist trained models to disk as .joblib files

    After training, runs 3 demo predictions to demonstrate how the
    predict() function works across different farm condition scenarios.

    Returns
    -------
    dict — results dict from train_models() (useful if called from another script)
    """
    print("=" * 52)
    print("  BETEL PALM PLANTING ADVISOR — ML PIPELINE")
    print("=" * 52)

    # ── Step 1: Generate Training Dataset ────────────────────────────
    # Creates 3,000 synthetic farm profiles with:
    #   - 8 input features (zone, soil, rainfall, temp, irrigation, region, humidity, elevation)
    #   - 3 output labels (season, drought risk, best shade crop)
    print("\n[1/4] Generating synthetic training dataset ...")
    df = generate_dataset(n=3000)

    # Print a quick sanity check on the dataset shape and class counts
    print(f"      Dataset shape   : {df.shape}")              # should be (3000, 11)
    print(f"      Season classes  : {dict(df['season_label'].value_counts())}")
    print(f"      Drought classes : {dict(df['drought_label'].value_counts())}")
    print(f"      Crop classes    : {dict(df['crop_label'].value_counts())}")

    # ── Step 2: Train All Three Models ───────────────────────────────
    # Each model is trained independently on the same feature set X
    # but targets a different output column (season / drought / crop)
    print("\n[2/4] Training Random Forest models ...")
    results = train_models(df)

    # ── Step 3: Generate Evaluation Charts ───────────────────────────
    # Produces a 4-row dashboard image:
    #   Row 0 = confusion matrices
    #   Row 1 = feature importance
    #   Row 2 = class distribution
    #   Row 3 = cross-validation accuracy comparison
    print("\n[3/4] Generating evaluation charts ...")
    chart_path = plot_results(
        results, df,
        out="/content/betel_palm_ml_report.png"
    )

    # ── Step 4: Save Trained Models ──────────────────────────────────
    # Saves each model as a .joblib file so they can be loaded later
    # without retraining (e.g. for deployment in a web app or API)
    print("\n[4/4] Saving trained models ...")
    save_models(results, path="/content/models/")

    # ── Demo Predictions ─────────────────────────────────────────────
    # These 3 test cases show how the predict() function behaves across
    # very different farm profiles.
    print("\n" + "=" * 52)
    print("  DEMO PREDICTIONS")
    print("=" * 52)

    test_cases = [
        # Scenario 1: Ideal conditions
        # Tropical zone, good loamy soil, full irrigation, South India
        # Expected: Early monsoon, Low drought risk, Banana as shade crop
        dict(zone="tropical", soil="loamy", rainfall_mm=1800, temp_c=28,
             irrigation="yes", region="india_south", humidity_pct=75, elevation_m=80,
             label="South India tropical farm"),

        # Scenario 2: Challenging conditions
        # Semiarid zone, sandy soil, no irrigation, very hot and dry
        # Expected: Post-monsoon, High drought risk, Tapioca as shade crop
        dict(zone="semiarid", soil="sandy", rainfall_mm=700, temp_c=36,
             irrigation="no", region="other", humidity_pct=40, elevation_m=200,
             label="Semi-arid, no irrigation"),

        # Scenario 3: Coastal alluvial
        # Coastal zone, fertile alluvial soil, partial irrigation, SE Asia
        # Expected: Pre-monsoon coastal, Low drought risk, Coconut as shade crop
        dict(zone="coastal", soil="alluvial", rainfall_mm=2000, temp_c=27,
             irrigation="partial", region="sea", humidity_pct=82, elevation_m=20,
             label="SE Asia coastal alluvial"),
    ]

    for case in test_cases:
        # Pop the display label out before passing the rest to predict()
        label = case.pop("label")
        print(f"\n  -- {label} --")

        # Run all three models on this farm profile
        preds = predict(results, **case)

        # Print the top prediction + confidence for each model
        for model_name, info in preds.items():
            key = model_name.replace("_label", "").replace("_", " ").title()
            print(f"    {key:<18}: {info['prediction']}  "
                  f"(confidence {info['confidence']})")

    print("\n" + "=" * 52)
    print("  Pipeline complete.")
    print(f"  Chart  -> betel_palm_ml_report.png")
    print(f"  Models -> models/")
    print("=" * 52)

    # Return results so the caller can use models directly without reloading
    return results


# ─────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────
# This block only executes when the script is run directly:
#   python betel_palm_ml.py
#
# If this file is imported as a module into another script (e.g. a web app),
# this block does NOT run automatically — the individual functions
# (predict, generate_dataset, train_models, etc.) can be imported and
# used independently without triggering the full training pipeline.

if __name__ == "__main__":
    main()


In [ ]:
results = main()
result = predict(
    models_dict  = results,
    zone         = "tropical",
    soil         = "loamy",
    rainfall_mm  = 1800,
    temp_c       = 28,
    irrigation   = "yes",
    region       = "india_south",
    humidity_pct = 75,
    elevation_m  = 80
)
print(result)

In [ ]:
import joblib
import numpy as np

# Load the drought risk model
drought_model = joblib.load('models/drought_label_rf.joblib')

# Prepare new data for prediction (example: semiarid, sandy, no irrigation)
# Using the same mapping as defined in the original script
# FEATURES = ["zone", "soil", "rainfall_mm", "temp_c", "irrigation", "region", "humidity_pct", "elevation_m"]
new_data_point = np.array([[
    ZONE_MAP["semiarid"],     # zone: semiarid
    SOIL_MAP["sandy"],        # soil: sandy
    700,                      # rainfall_mm: 700
    36,                       # temp_c: 36
    IRR_MAP["no"],            # irrigation: no
    REGION_MAP["other"],      # region: other
    40,                       # humidity_pct: 40
    200                       # elevation_m: 200
]])

# Make a prediction
prediction_idx = drought_model.predict(new_data_point)[0]
predicted_label = DROUGHT_LABELS[prediction_idx]

# Get probabilities
probabilities = drought_model.predict_proba(new_data_point)[0]
confidence = f"{probabilities.max()*100:.1f}%"

print(f"Predicted Drought Risk: {predicted_label} (confidence: {confidence})")

**ML model using lattitude and longitude**

In [ ]:
"""
Betel Palm (Areca catechu) Planting Advisor — Lat/Lon Edition
=============================================================
This version extends the original pipeline with coordinate-based prediction.

NEW: predict_from_coords(lat, lon, irrigation="partial")
     Pass any GPS latitude/longitude and the model will:
       1. Identify the geographic region from coordinates
       2. Estimate climate zone, rainfall, temperature, humidity, elevation
       3. Run all three trained models and return recommendations

Models trained:
  1. season_label   → best planting season
  2. drought_label  → drought / drying risk level
  3. crop_label     → best companion shade crop

Run:
    python betel_palm_ml_latlon.py

Output:
    betel_palm_latlon_report.png   — evaluation charts
    models/                        — saved .joblib model files
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings, os, joblib, math
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score

np.random.seed(42)


# ═══════════════════════════════════════════════════════════════════
# SECTION 1 — ENCODING MAPS & CONSTANTS
# ═══════════════════════════════════════════════════════════════════

# Categorical inputs → integer codes for the ML model
ZONE_MAP   = {"tropical": 0, "subtropical": 1, "semiarid": 2, "coastal": 3}
SOIL_MAP   = {"loamy": 0, "clay": 1, "sandy": 2, "laterite": 3, "alluvial": 4}
IRR_MAP    = {"yes": 0, "partial": 1, "no": 2}
REGION_MAP = {"india_south": 0, "india_ne": 1, "sea": 2, "pacific": 3,
              "africa": 4, "latam": 5, "other": 6}

# Human-readable output labels (index = class integer predicted by the model)
SEASON_LABELS  = ["Early monsoon (May-Aug)", "Summer monsoon (Jun-Sep)",
                  "Post-monsoon (Jul-Oct)",  "Pre-monsoon coastal (Apr-Jul)"]
DROUGHT_LABELS = ["Low risk", "Medium risk", "High risk"]
CROP_LABELS    = ["Banana", "Coconut", "Tapioca/Cassava",
                  "Black pepper", "Turmeric", "Jackfruit"]

# Feature columns fed into every model — ORDER IS CRITICAL
# Latitude and longitude are included so the model learns spatial patterns directly
FEATURES = [
    "latitude",      # GPS latitude  (-90 to +90)
    "longitude",     # GPS longitude (-180 to +180)
    "zone",          # climate zone encoded as int (0-3)
    "soil",          # soil type encoded as int (0-4)
    "rainfall_mm",   # estimated annual rainfall in mm
    "temp_c",        # estimated average temperature in °C
    "irrigation",    # irrigation availability encoded as int (0-2)
    "region",        # geographic region encoded as int (0-6)
    "humidity_pct",  # estimated relative humidity %
    "elevation_m",   # estimated elevation in metres
]


# ═══════════════════════════════════════════════════════════════════
# SECTION 2 — COORDINATE → CLIMATE ESTIMATOR
# ═══════════════════════════════════════════════════════════════════

# This is the core new section. It encodes climatological knowledge
# about how latitude, longitude, and proximity to coast relate to
# temperature, rainfall, humidity and growing conditions.

# Coastal proximity threshold in degrees (~111 km per degree)
COASTAL_DEG = 3.0

def _distance_to_coast(lat, lon):
    """
    Estimate the minimum angular distance from a coordinate to the
    nearest ocean coastline using a simplified lookup of major coastlines.

    This is a fast approximation — not a full GIS calculation.
    It checks distance to a grid of known coastal points and returns
    the closest one in degrees.

    Parameters
    ----------
    lat : float — latitude
    lon : float — longitude

    Returns
    -------
    float — approximate degrees to nearest coastal point
    """
    # Representative coastal reference points (lat, lon) around the world
    # Covers: Indian subcontinent, SE Asia, Pacific, Africa east coast, Americas
    coastal_points = [
        # Indian Ocean coasts
        (8.5, 76.9),   # Kerala coast, South India
        (13.1, 80.3),  # Tamil Nadu coast
        (19.9, 86.1),  # Odisha coast
        (6.9, 79.8),   # Sri Lanka west coast
        (7.9, 81.7),   # Sri Lanka east coast
        (22.1, 91.8),  # Bangladesh coast
        (17.7, 83.3),  # Andhra Pradesh coast
        # SE Asia
        (1.3, 103.8),  # Singapore
        (13.7, 100.5), # Thailand Gulf coast
        (16.8, 96.1),  # Myanmar coast
        (10.8, 106.7), # Vietnam south coast
        (3.1, 101.7),  # Malaysia west coast
        # Pacific
        (-8.5, 115.2), # Bali, Indonesia
        (-6.2, 106.8), # Java, Indonesia
        (14.5, 121.0), # Philippines
        (-17.7, 168.3),# Vanuatu
        # East Africa
        (-4.0, 39.7),  # Kenya coast
        (-11.7, 43.3), # Comoros
        (1.3, 44.3),   # Somalia coast
        # West Africa
        (5.6, -0.2),   # Ghana coast
        (6.4, 2.4),    # Benin coast
        # Americas
        (10.5, -61.4), # Trinidad
        (-2.5, -44.3), # Brazil northeast coast
        (8.9, -79.5),  # Panama coast
        (10.5, -66.9), # Venezuela coast
    ]

    # Find the minimum angular distance to any coastal reference point
    # Using Euclidean approximation (good enough for small-to-medium distances)
    min_dist = float("inf")
    for clat, clon in coastal_points:
        # Scale longitude by cos(lat) to account for meridian convergence
        dlat = lat - clat
        dlon = (lon - clon) * math.cos(math.radians(lat))
        dist = math.sqrt(dlat**2 + dlon**2)
        if dist < min_dist:
            min_dist = dist
    return min_dist


def coords_to_features(lat, lon, irrigation="partial"):
    """
    Convert GPS coordinates to a full set of agronomic farm features.

    This function encodes climatological knowledge about how geographic
    position relates to climate zone, rainfall, temperature, and humidity.

    Key rules applied:
      - Latitude band determines base climate zone (tropical vs subtropical etc.)
      - Proximity to coast adjusts zone to "coastal" and boosts humidity/rainfall
      - Longitude ranges identify known agricultural regions (South India, SE Asia, etc.)
      - Elevation is estimated from lat/lon using a simplified terrain model
      - Temperature is estimated as: 30 - 0.6 * abs(lat) (lapse with latitude)
      - Rainfall is estimated from zone + regional patterns + coastal proximity

    Parameters
    ----------
    lat        : float — GPS latitude  (negative = southern hemisphere)
    lon        : float — GPS longitude (negative = western hemisphere)
    irrigation : str   — "yes" | "partial" | "no"  (user must supply this)

    Returns
    -------
    dict — all 10 features ready to pass into the model, plus:
           "zone_name"   (str) — human-readable climate zone
           "region_name" (str) — human-readable region name
           "notes"       (list of str) — explanation of each derived value
    """
    notes = []   # collects human-readable explanations of each decision

    # ── Step 1: Identify geographic region from lon/lat bands ─────────
    # These bounding boxes cover the main betel palm growing regions.
    # Checked in priority order — first match wins.

    if 5 <= lat <= 25 and 68 <= lon <= 82:
        region_name = "india_south"
        notes.append(f"Region: South India / Sri Lanka (lat {lat:.1f}, lon {lon:.1f})")
    elif 20 <= lat <= 30 and 88 <= lon <= 97:
        region_name = "india_ne"
        notes.append(f"Region: Northeast India / Bangladesh")
    elif -10 <= lat <= 25 and 93 <= lon <= 141:
        region_name = "sea"
        notes.append(f"Region: Southeast Asia (Thailand, Myanmar, Vietnam, Indonesia)")
    elif -25 <= lat <= 25 and 25 <= lon <= 52:
        region_name = "africa"
        notes.append(f"Region: East / Central Africa")
    elif -10 <= lat <= 20 and -80 <= lon <= -35:
        region_name = "latam"
        notes.append(f"Region: Latin America / Caribbean")
    elif -30 <= lat <= 30 and 130 <= lon <= 180:
        region_name = "pacific"
        notes.append(f"Region: Pacific Islands")
    else:
        region_name = "other"
        notes.append(f"Region: Other tropical / subtropical area")

    # ── Step 2: Determine climate zone from latitude band ─────────────
    # Betel palm grows between roughly 30°S and 30°N.
    # abs(lat) gives distance from equator regardless of hemisphere.
    abs_lat = abs(lat)

    if abs_lat <= 10:
        base_zone = "tropical"
        notes.append(f"Zone: Tropical (latitude {lat:.1f}° is within 10° of equator)")
    elif abs_lat <= 23.5:
        base_zone = "tropical"   # still tropical within the tropics band
        notes.append(f"Zone: Tropical (within Tropics of Cancer/Capricorn)")
    elif abs_lat <= 30:
        base_zone = "subtropical"
        notes.append(f"Zone: Subtropical (latitude {lat:.1f}° is 23.5°-30° from equator)")
    else:
        base_zone = "subtropical"
        notes.append(f"Zone: Subtropical/marginal (latitude {lat:.1f}° — borderline for betel palm)")

    # ── Step 3: Coastal override ──────────────────────────────────────
    # If the farm is close to the coast, override zone to "coastal"
    # regardless of latitude band, as sea influence dominates microclimate.
    coast_dist = _distance_to_coast(lat, lon)
    is_coastal = coast_dist < COASTAL_DEG

    if is_coastal:
        zone_name = "coastal"
        notes.append(f"Zone overridden to Coastal (approx {coast_dist*111:.0f} km from coast)")
    else:
        zone_name = base_zone

    # ── Step 4: Semiarid override for known dry regions ───────────────
    # Certain longitude/latitude bands are climatologically semiarid
    # even if the latitude would suggest tropical/subtropical.
    semiarid_regions = [
        (10, 25, 68, 78),    # Deccan plateau, interior India
        (-5, 15, 33, 43),    # East African interior (Ethiopia, Kenya highlands)
        (-30, 0, -65, -35),  # Brazilian sertao / Caatinga
    ]
    for la1, la2, lo1, lo2 in semiarid_regions:
        if la1 <= lat <= la2 and lo1 <= lon <= lo2 and not is_coastal:
            zone_name = "semiarid"
            notes.append(f"Zone overridden to Semiarid (known dry interior region)")
            break

    # ── Step 5: Estimate average annual temperature ───────────────────
    # Base rule: temperature decreases by ~0.6°C per degree of latitude from equator.
    # Additional adjustment for known hot interior regions.
    base_temp = 30 - (abs_lat * 0.55)

    if zone_name == "semiarid":
        base_temp += 3      # dry interiors are hotter
    elif is_coastal:
        base_temp -= 1.5    # sea moderates temperature

    temp_c = int(np.clip(base_temp, 19, 41))
    notes.append(f"Estimated temperature: {temp_c}°C")

    # ── Step 6: Estimate annual rainfall ─────────────────────────────
    # Base rainfall determined by climate zone, then adjusted for:
    #   - coastal proximity (more rainfall near coast)
    #   - specific high-rainfall regions (Western Ghats, SE Asia)
    #   - specific dry regions

    if zone_name == "tropical":
        base_rain = 1800
    elif zone_name == "coastal":
        base_rain = 2000
    elif zone_name == "subtropical":
        base_rain = 1100
    elif zone_name == "semiarid":
        base_rain = 650
    else:
        base_rain = 1200

    # Regional rainfall adjustments based on known climatology
    if region_name == "sea":
        base_rain += 400    # SE Asia is one of the wettest regions on earth
        notes.append("Rainfall boosted: SE Asia has high precipitation")
    if region_name == "india_ne":
        base_rain += 300    # Northeast India / Bangladesh very wet
        notes.append("Rainfall boosted: Northeast India high monsoon")
    if region_name == "india_south":
        # Western Ghats boost (west coast gets more, east coast drier)
        if lon < 77:
            base_rain += 200
            notes.append("Rainfall boosted: West coast / Western Ghats influence")
        else:
            base_rain -= 100
            notes.append("Rainfall slightly reduced: leeward (east) side")

    # Coastal proximity adds rainfall (orographic + sea-moisture effect)
    if is_coastal:
        coast_bonus = max(0, int((COASTAL_DEG - coast_dist) * 150))
        base_rain += coast_bonus

    rainfall_mm = int(np.clip(base_rain, 400, 3600))
    notes.append(f"Estimated annual rainfall: {rainfall_mm} mm")

    # ── Step 7: Estimate relative humidity ───────────────────────────
    if zone_name == "tropical":
        humidity = 72
    elif zone_name == "coastal":
        humidity = 80
    elif zone_name == "subtropical":
        humidity = 60
    elif zone_name == "semiarid":
        humidity = 42
    else:
        humidity = 62

    if is_coastal:
        humidity = min(humidity + 10, 95)
    if region_name == "sea":
        humidity = min(humidity + 8, 95)

    humidity_pct = int(humidity)
    notes.append(f"Estimated humidity: {humidity_pct}%")

    # ── Step 8: Estimate elevation ────────────────────────────────────
    # Very rough estimate — lowland coastal areas near 0m,
    # interior plateaus/highlands modelled from known terrain patterns.
    if is_coastal:
        elevation = int(np.clip(np.random.normal(30, 20), 0, 150))
    elif zone_name == "semiarid":
        elevation = int(np.clip(np.random.normal(350, 100), 50, 600))
    else:
        elevation = int(np.clip(np.random.normal(100, 60), 0, 500))
    notes.append(f"Estimated elevation: {elevation} m")

    # ── Step 9: Infer likely soil type from region + zone ─────────────
    # This is a heuristic — ideally the farmer would supply actual soil type.
    if region_name == "india_south":
        soil_name = "laterite"   # red laterite soils dominate South India
        notes.append("Soil estimated as laterite (typical South India)")
    elif region_name in ("sea", "pacific"):
        soil_name = "alluvial"   # river deltas dominate SE Asia agriculture
        notes.append("Soil estimated as alluvial (typical SE Asia river basin)")
    elif zone_name == "semiarid":
        soil_name = "sandy"      # semiarid zones have sandy / sandy-loam soils
        notes.append("Soil estimated as sandy (typical semiarid zone)")
    elif is_coastal:
        soil_name = "alluvial"   # coastal floodplains are typically alluvial
        notes.append("Soil estimated as alluvial (coastal floodplain)")
    else:
        soil_name = "loamy"      # default — most common agricultural soil
        notes.append("Soil estimated as loamy (default)")

    # ── Assemble the complete feature dictionary ──────────────────────
    return {
        # Raw GPS coordinates — also used as features by the model
        "latitude":     round(lat, 4),
        "longitude":    round(lon, 4),

        # Derived categorical features (encoded as integers)
        "zone":         ZONE_MAP[zone_name],
        "soil":         SOIL_MAP[soil_name],
        "irrigation":   IRR_MAP[irrigation],
        "region":       REGION_MAP[region_name],

        # Derived numeric climate estimates
        "rainfall_mm":  rainfall_mm,
        "temp_c":       temp_c,
        "humidity_pct": humidity_pct,
        "elevation_m":  elevation,

        # Human-readable equivalents for display
        "zone_name":    zone_name,
        "region_name":  region_name,
        "soil_name":    soil_name,
        "notes":        notes,
    }


# ═══════════════════════════════════════════════════════════════════
# SECTION 3 — DOMAIN RULE ENGINES (unchanged logic, extended region map)
# ═══════════════════════════════════════════════════════════════════

def _season_label(zone, rainfall, irrigation, temp):
    """Best planting season from climate zone."""
    if zone == 0:   return 0   # tropical → early monsoon May-Aug
    elif zone == 1: return 1   # subtropical → summer monsoon Jun-Sep
    elif zone == 2: return 2   # semiarid → post-monsoon Jul-Oct
    else:           return 3   # coastal → pre-monsoon coastal Apr-Jul


def _drought_label(zone, soil, rainfall, irrigation, temp, humidity):
    """
    Drought risk score.
    Each risk condition adds points; protective conditions subtract.
    Final score: 0-2 = Low, 3-5 = Medium, 6+ = High.
    """
    score = 0
    if soil == 2:          score += 2   # sandy soil loses moisture fast
    if irrigation == 2:    score += 2   # no irrigation (rain-fed only)
    if zone == 2:          score += 2   # semiarid — structurally dry region
    if rainfall < 800:     score += 2   # critically low annual rainfall
    elif rainfall < 1200:  score += 1   # moderately low rainfall
    if temp > 35:          score += 1   # heat stress increases evaporation
    if humidity < 50:      score += 1   # dry air accelerates moisture loss
    if soil in (0, 4):     score -= 1   # loamy / alluvial holds moisture well
    if irrigation == 0:    score -= 1   # has irrigation to top up water
    score = max(score, 0)
    if score <= 2:   return 0   # Low
    elif score <= 5: return 1   # Medium
    else:            return 2   # High


def _crop_label(zone, soil, region, temp, rainfall, irrigation):
    """
    Best shade / companion crop using a points-based system.
    Each crop scores points for matching farm conditions.
    The highest-scoring crop wins.
    """
    scores = {"Banana": 0, "Coconut": 0, "Tapioca/Cassava": 0,
              "Black pepper": 0, "Turmeric": 0, "Jackfruit": 0}

    if zone in (0, 1) and rainfall > 1200:  scores["Banana"] += 3
    if zone in (0, 3):                       scores["Coconut"] += 3
    if zone == 2:                            scores["Tapioca/Cassava"] += 4
    if region in (0, 1, 2):                  scores["Black pepper"] += 3   # india + sea
    if soil in (0, 4):                       scores["Turmeric"] += 2
    if zone in (0, 1):                       scores["Jackfruit"] += 2
    if rainfall < 900:
        scores["Tapioca/Cassava"] += 2
        scores["Banana"] -= 2
    if irrigation == 0:                      scores["Banana"] += 1

    return CROP_LABELS.index(max(scores, key=scores.get))


# ═══════════════════════════════════════════════════════════════════
# SECTION 4 — SYNTHETIC DATASET GENERATION (lat/lon aware)
# ═══════════════════════════════════════════════════════════════════

# Known betel palm growing regions with representative lat/lon ranges
# Format: (lat_min, lat_max, lon_min, lon_max, zone, region, rainfall_centre)
GROWING_REGIONS = [
    # South India / Sri Lanka — largest growing region globally
    (5.0,  22.0, 68.0,  82.0, 0, 0, 1600),
    # Northeast India / Bangladesh — heavy monsoon region
    (20.0, 28.0, 88.0,  96.0, 0, 1, 2000),
    # SE Asia (Thailand, Myanmar, Indonesia, Philippines)
    (-8.0, 22.0, 95.0, 138.0, 0, 2, 2200),
    # Pacific Islands (PNG, Fiji, Vanuatu)
    (-20.0, 0.0, 140.0, 178.0, 0, 3, 2400),
    # East Africa (Kenya, Tanzania, Mozambique)
    (-15.0, 5.0,  33.0,  42.0, 0, 4, 1000),
    # Latin America (Brazil, Ecuador, Colombia)
    (-10.0, 10.0, -78.0, -35.0, 0, 5, 1800),
]

def generate_dataset(n=3000):
    """
    Generate a synthetic labelled dataset of n farm profiles.

    Each sample is assigned to a real growing region and gets a
    random lat/lon within that region. Climate features are then
    derived from those coordinates using coords_to_features().

    Parameters
    ----------
    n : int — number of synthetic farm profiles to generate

    Returns
    -------
    pd.DataFrame — shape (n, 13) with FEATURES + 3 label columns
    """
    rows = []

    for _ in range(n):

        # ── Pick a growing region (weighted towards South India and SE Asia)
        # These are the two largest commercial betel palm regions globally.
        region_idx = np.random.choice(
            len(GROWING_REGIONS),
            p=[0.30, 0.18, 0.25, 0.10, 0.10, 0.07]
        )
        la1, la2, lo1, lo2, zone_hint, reg_hint, rain_hint = GROWING_REGIONS[region_idx]

        # Sample a random lat/lon within the chosen region's bounding box
        lat = round(np.random.uniform(la1, la2), 3)
        lon = round(np.random.uniform(lo1, lo2), 3)

        # Sample irrigation availability (40% full, 30% partial, 30% rain-fed)
        irr_str = np.random.choice(["yes", "partial", "no"], p=[0.40, 0.30, 0.30])

        # Derive all climate features from the coordinates
        feats = coords_to_features(lat, lon, irrigation=irr_str)

        # Add small random variation to climate estimates for realism
        rainfall  = int(np.clip(feats["rainfall_mm"]  + np.random.normal(0, 150), 400, 3600))
        temp      = int(np.clip(feats["temp_c"]        + np.random.normal(0, 2),   19, 41))
        humidity  = int(np.clip(feats["humidity_pct"]  + np.random.normal(0, 8),   30, 98))
        elevation = int(np.clip(feats["elevation_m"]   + np.random.normal(0, 40),  0,  600))

        zone       = feats["zone"]
        soil       = feats["soil"]
        irrigation = feats["irrigation"]
        region     = feats["region"]

        # Generate target labels using the domain rule engines
        season  = _season_label(zone, rainfall, irrigation, temp)
        drought = _drought_label(zone, soil, rainfall, irrigation, temp, humidity)
        crop    = _crop_label(zone, soil, region, temp, rainfall, irrigation)

        # Inject 7% label noise to simulate real-world variability
        if np.random.random() < 0.07: season  = np.random.randint(0, 4)
        if np.random.random() < 0.07: drought = np.random.randint(0, 3)

        # Append one row: 10 features + 3 labels
        rows.append([lat, lon, zone, soil, rainfall, temp,
                     irrigation, region, humidity, elevation,
                     season, drought, crop])

    df = pd.DataFrame(
        rows,
        columns=FEATURES + ["season_label", "drought_label", "crop_label"]
    )
    return df


# ═══════════════════════════════════════════════════════════════════
# SECTION 5 — MODEL TRAINING
# ═══════════════════════════════════════════════════════════════════

def train_models(df):
    """
    Train three Random Forest classifiers on the lat/lon-enriched dataset.

    Each model trains on all 10 features including latitude and longitude.
    Including coordinates allows the model to learn spatial patterns like:
      - "High latitudes tend to have lower rainfall"
      - "SE Asian longitudes correlate with better shade crop options"
      - "Coastal latitudes near the equator have low drought risk"

    Parameters
    ----------
    df : pd.DataFrame — output from generate_dataset()

    Returns
    -------
    dict — trained model + evaluation metrics for each target
    """
    X = df[FEATURES].values   # shape (n, 10) — 10 features including lat/lon
    results = {}

    for target, label_list, clf_kw in [
        ("season_label",  SEASON_LABELS,  dict(n_estimators=200, max_depth=8,  min_samples_leaf=4)),
        ("drought_label", DROUGHT_LABELS, dict(n_estimators=200, max_depth=6,  min_samples_leaf=4)),
        ("crop_label",    CROP_LABELS,    dict(n_estimators=200, max_depth=10, min_samples_leaf=3)),
    ]:
        y = df[target].values

        # 80/20 stratified train/test split
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.20, random_state=42, stratify=y
        )

        # Random Forest with balanced class weights for imbalanced classes
        model = RandomForestClassifier(
            random_state=42, n_jobs=-1, class_weight="balanced", **clf_kw
        )
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_te)
        acc    = accuracy_score(y_te, y_pred)

        # 5-fold cross-validation to confirm the model generalises
        cv = cross_val_score(
            model, X, y,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
            scoring="accuracy", n_jobs=-1
        )

        results[target] = {
            "model":   model,
            "X_te":    X_te,
            "y_te":    y_te,
            "y_pred":  y_pred,
            "acc":     acc,
            "cv_mean": cv.mean(),
            "cv_std":  cv.std(),
            "labels":  label_list,
        }

        print(f"\n{'─'*52}")
        print(f"  Model : {target}")
        print(f"  Test accuracy : {acc*100:.1f}%")
        print(f"  CV accuracy   : {cv.mean()*100:.1f}% +/- {cv.std()*100:.1f}%")
        # Use only the classes present in the test set for the report
        present = sorted(np.unique(np.concatenate([y_te, y_pred])))
        present_names = [label_list[i] for i in present]
        print(f"\n{classification_report(y_te, y_pred, labels=present, target_names=present_names)}")

    return results


# ═══════════════════════════════════════════════════════════════════
# SECTION 6 — PREDICT FROM COORDINATES  ★ NEW MAIN FUNCTION ★
# ═══════════════════════════════════════════════════════════════════

def predict_from_coords(models_dict, lat, lon, irrigation="partial", verbose=True):
    """
    Predict betel palm planting recommendations from GPS coordinates.

    This is the main new function. It:
      1. Calls coords_to_features(lat, lon) to estimate all climate parameters
      2. Assembles the feature vector in the correct model input order
      3. Runs all three trained models
      4. Returns predictions with confidence and the derived climate context

    Parameters
    ----------
    models_dict : dict  — output from train_models()
    lat         : float — GPS latitude  (e.g. 10.5 for Kerala, India)
    lon         : float — GPS longitude (e.g. 76.2 for Kerala, India)
    irrigation  : str   — "yes" | "partial" | "no"
                          This is the ONE parameter the user must supply manually
                          because it depends on farm infrastructure, not location.
    verbose     : bool  — if True, prints a formatted report to the console

    Returns
    -------
    dict with keys:
      "coordinates"  : {"lat": float, "lon": float}
      "derived"      : dict of all climate parameters estimated from coordinates
      "predictions"  : dict with season, drought, crop predictions + probabilities
      "notes"        : list of str explaining how each parameter was estimated

    Example
    -------
    result = predict_from_coords(results, lat=10.5, lon=76.2, irrigation="yes")
    print(result["predictions"]["season_label"]["prediction"])
    # "Early monsoon (May-Aug)"
    """
    # ── Step 1: Derive climate features from coordinates ─────────────
    feats = coords_to_features(lat, lon, irrigation=irrigation)

    # ── Step 2: Assemble feature vector in correct model input order ──
    # Order MUST match the FEATURES list defined at the top of this file.
    x = np.array([[
        feats["latitude"],
        feats["longitude"],
        feats["zone"],
        feats["soil"],
        feats["rainfall_mm"],
        feats["temp_c"],
        feats["irrigation"],
        feats["region"],
        feats["humidity_pct"],
        feats["elevation_m"],
    ]])
    # x.shape = (1, 10) — one sample, ten features

    # ── Step 3: Run all three models ──────────────────────────────────
    predictions = {}
    for key, labels in [
        ("season_label",  SEASON_LABELS),
        ("drought_label", DROUGHT_LABELS),
        ("crop_label",    CROP_LABELS),
    ]:
        m     = models_dict[key]["model"]
        pred  = m.predict(x)[0]          # most likely class index
        proba = m.predict_proba(x)[0]    # probability per class

        predictions[key] = {
            "prediction":    labels[pred],
            "confidence":    f"{proba.max() * 100:.1f}%",
            "probabilities": {labels[i]: f"{p*100:.1f}%" for i, p in enumerate(proba)},
        }

    # ── Step 4: Build the full result dictionary ──────────────────────
    result = {
        "coordinates": {"lat": lat, "lon": lon},
        "derived": {
            "zone":         feats["zone_name"],
            "region":       feats["region_name"],
            "soil":         feats["soil_name"],
            "rainfall_mm":  feats["rainfall_mm"],
            "temp_c":       feats["temp_c"],
            "humidity_pct": feats["humidity_pct"],
            "elevation_m":  feats["elevation_m"],
            "irrigation":   irrigation,
        },
        "predictions": predictions,
        "notes":        feats["notes"],
    }

    # ── Step 5: Pretty-print if verbose=True ──────────────────────────
    if verbose:
        print(f"\n{'═'*56}")
        print(f"  BETEL PALM PREDICTION — Lat {lat}, Lon {lon}")
        print(f"{'═'*56}")
        print(f"\n  DERIVED CLIMATE CONTEXT")
        print(f"  {'─'*48}")
        for note in feats["notes"]:
            print(f"    {note}")
        print(f"\n  DERIVED PARAMETERS")
        print(f"  {'─'*48}")
        d = result["derived"]
        print(f"    Zone        : {d['zone']}")
        print(f"    Region      : {d['region']}")
        print(f"    Soil (est.) : {d['soil']}")
        print(f"    Rainfall    : {d['rainfall_mm']} mm/year")
        print(f"    Temperature : {d['temp_c']} °C avg")
        print(f"    Humidity    : {d['humidity_pct']} %")
        print(f"    Elevation   : {d['elevation_m']} m")
        print(f"    Irrigation  : {irrigation}")
        print(f"\n  RECOMMENDATIONS")
        print(f"  {'─'*48}")
        for model_key, info in predictions.items():
            label = model_key.replace("_label","").replace("_"," ").title()
            print(f"    {label:<18}: {info['prediction']:<35} (confidence {info['confidence']})")
        print(f"{'═'*56}\n")

    return result


# ═══════════════════════════════════════════════════════════════════
# SECTION 7 — EVALUATION CHARTS
# ═══════════════════════════════════════════════════════════════════

def plot_results(results, df, out="betel_palm_latlon_report.png"):
    """
    Generate a 4-row evaluation dashboard and save as PNG.

    Row 0 — Confusion matrices   : actual vs predicted heatmaps
    Row 1 — Feature importance   : which of the 10 features matter most
    Row 2 — Geographic scatter   : training samples plotted on lat/lon axes
    Row 3 — CV accuracy bars     : model comparison
    """
    fig = plt.figure(figsize=(20, 22), facecolor="#F8F8F5")
    fig.suptitle("Betel Palm Advisor — Lat/Lon ML Model Report",
                 fontsize=20, fontweight="bold", y=0.99, color="#1a1a1a")

    gs = gridspec.GridSpec(4, 3, figure=fig,
                           hspace=0.55, wspace=0.38, top=0.96, bottom=0.03)

    COLORS       = ["#3B6D11", "#185FA5", "#854F0B"]
    target_names = ["season_label", "drought_label", "crop_label"]
    titles       = ["Planting Season", "Drought Risk", "Shade Crop"]

    # ── Row 0: Confusion matrices ──────────────────────────────────────
    for col, (key, color) in enumerate(zip(target_names, COLORS)):
        r  = results[key]
        ax = fig.add_subplot(gs[0, col])
        present = sorted(np.unique(np.concatenate([r["y_te"], r["y_pred"]])))
        present_labels = [r["labels"][i] for i in present]
        cm = confusion_matrix(r["y_te"], r["y_pred"], labels=present)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=present_labels)
        disp.plot(ax=ax, colorbar=False,
                  cmap="Greens" if col == 0 else "Blues" if col == 1 else "Oranges")
        ax.set_title(
            f"{titles[col]}\nTest: {r['acc']*100:.1f}%  CV: {r['cv_mean']*100:.1f}%+/-{r['cv_std']*100:.1f}%",
            fontsize=9, pad=5, color="#222")
        ax.tick_params(axis="x", labelsize=6, rotation=30)
        ax.tick_params(axis="y", labelsize=6)
        for t in ax.texts: t.set_fontsize(7)

    # ── Row 1: Feature importance (now shows lat/lon among top features) ─
    for col, (key, color) in enumerate(zip(target_names, COLORS)):
        r   = results[key]
        ax  = fig.add_subplot(gs[1, col])
        imp = r["model"].feature_importances_
        idx = np.argsort(imp)
        ax.barh([FEATURES[i] for i in idx], imp[idx], color=color, alpha=0.82)
        ax.set_title(f"Feature importance — {titles[col]}", fontsize=9)
        ax.tick_params(axis="both", labelsize=8)
        ax.set_xlabel("Mean decrease in impurity", fontsize=7)
        ax.spines[["top","right"]].set_visible(False)

    # ── Row 2: Geographic scatter — training samples on lat/lon axes ───
    # This plot shows the spatial distribution of the training data.
    # Colour encodes the target class for each model.
    zone_colors = ["#1D9E75","#185FA5","#D85A30","#3B6D11"]   # one per season class
    drought_colors = ["#1D9E75","#EF9F27","#E24B4A"]           # green/amber/red
    crop_palette = ["#3B6D11","#185FA5","#D85A30","#533AB7","#854F0B","#3B6D11"]

    scatter_config = [
        ("season_label",  SEASON_LABELS,  zone_colors),
        ("drought_label", DROUGHT_LABELS, drought_colors),
        ("crop_label",    CROP_LABELS,    crop_palette),
    ]
    for col, (key, labels, palette) in enumerate(scatter_config):
        ax = fig.add_subplot(gs[2, col])
        for cls_idx, (lbl, clr) in enumerate(zip(labels, palette)):
            mask = df[key] == cls_idx
            ax.scatter(df.loc[mask, "longitude"], df.loc[mask, "latitude"],
                       c=clr, s=4, alpha=0.5, label=lbl)
        ax.set_xlabel("Longitude", fontsize=8)
        ax.set_ylabel("Latitude",  fontsize=8)
        ax.set_title(f"Training data geography — {titles[col]}", fontsize=9)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=6, markerscale=2, loc="lower right")
        ax.spines[["top","right"]].set_visible(False)

    # ── Row 3: CV accuracy comparison ──────────────────────────────────
    ax_cv     = fig.add_subplot(gs[3, :])
    models_cv = [(t, results[t]["cv_mean"]*100, results[t]["cv_std"]*100) for t in target_names]
    bars = ax_cv.bar(
        [t[0].replace("_label","").replace("_"," ").title() for t in models_cv],
        [t[1] for t in models_cv],
        yerr=[t[2] for t in models_cv],
        color=COLORS, alpha=0.82, capsize=6, width=0.45,
        error_kw=dict(elinewidth=1.2, ecolor="#555")
    )
    ax_cv.set_ylim(0, 110)
    ax_cv.set_ylabel("5-Fold CV Accuracy (%)", fontsize=10)
    ax_cv.set_title("Cross-Validation Accuracy — All Three Models (with Lat/Lon features)", fontsize=11)
    ax_cv.spines[["top","right"]].set_visible(False)
    ax_cv.axhline(90, color="#aaa", lw=0.8, ls="--")
    ax_cv.text(2.6, 91, "90% target", fontsize=8, color="#888")
    for bar, (_, mean, std) in zip(bars, models_cv):
        ax_cv.text(bar.get_x() + bar.get_width()/2,
                   bar.get_height() + std + 1,
                   f"{mean:.1f}%", ha="center", fontsize=10, fontweight="bold")

    plt.savefig(out, dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor())
    print(f"\n  Chart saved -> {out}")
    return out


# ═══════════════════════════════════════════════════════════════════
# SECTION 8 — SAVE MODELS
# ═══════════════════════════════════════════════════════════════════

def save_models(results, path="models"):
    """Save all three trained models to disk as .joblib files."""
    os.makedirs(path, exist_ok=True)
    for key in ("season_label", "drought_label", "crop_label"):
        fpath = os.path.join(path, f"{key}_latlon_rf.joblib")
        joblib.dump(results[key]["model"], fpath)
    print(f"\n  Models saved -> {path}/")


# ═══════════════════════════════════════════════════════════════════
# SECTION 9 — MAIN PIPELINE
# ═══════════════════════════════════════════════════════════════════

def main():
    print("=" * 56)
    print("  BETEL PALM ADVISOR — LAT/LON ML PIPELINE")
    print("=" * 56)

    # Step 1: Generate lat/lon-aware training dataset
    print("\n[1/4] Generating geo-aware synthetic training dataset ...")
    df = generate_dataset(n=3000)
    print(f"      Dataset shape   : {df.shape}")
    print(f"      Lat range       : {df['latitude'].min():.1f} to {df['latitude'].max():.1f}")
    print(f"      Lon range       : {df['longitude'].min():.1f} to {df['longitude'].max():.1f}")
    print(f"      Season classes  : {dict(df['season_label'].value_counts())}")
    print(f"      Drought classes : {dict(df['drought_label'].value_counts())}")

    # Step 2: Train all three models (lat + lon now included as features)
    print("\n[2/4] Training models with lat/lon features ...")
    results = train_models(df)

    # Step 3: Generate evaluation charts
    print("\n[3/4] Generating evaluation charts ...")
    plot_results(results, df, out="/content/betel_palm_ml_report.png")

    # Step 4: Save trained models to disk
    print("\n[4/4] Saving models ...")
    save_models(results, path="/content/models/")

    # ── Demo predictions using real-world coordinates ─────────────────
    print("\n" + "=" * 56)
    print("  DEMO — PREDICT FROM COORDINATES")
    print("=" * 56)

    # demo_locations = [
    #     # (lat,    lon,    irrigation, description)
    #     (10.52,  76.21,  "yes",      "Thrissur, Kerala, India — betel palm heartland"),
    #     (13.08,  80.27,  "partial",  "Chennai, Tamil Nadu, India — coastal city"),
    #     (15.32,  73.87,  "yes",      "Goa, India — coastal tropical"),
    #     (18.52,  73.86,  "partial",  "Pune, India — interior Deccan plateau"),
    #     (6.93,   79.85,  "yes",      "Colombo, Sri Lanka — coastal tropical"),
    #     (13.75, 100.52,  "partial",  "Bangkok, Thailand — SE Asia"),
    #     (-6.21, 106.85,  "partial",  "Jakarta, Indonesia — equatorial coastal"),
    #     (-8.65, 115.22,  "no",       "Bali, Indonesia — tropical island"),
    #     (-4.04,  39.67,  "no",       "Mombasa, Kenya — East African coast"),
    #     (3.14,  101.69,  "yes",      "Kuala Lumpur, Malaysia"),
    # ]

    demo_locations = [
        # (lat,    lon,    irrigation, description)
        (10.52,  76.21,  "yes",      "Thrissur, Kerala, India — betel palm heartland"),
    ]

    for lat, lon, irr, desc in demo_locations:
        print(f"\n  Location: {desc}")
        predict_from_coords(results, lat=lat, lon=lon, irrigation=irr, verbose=True)

    print("=" * 56)
    print("  Pipeline complete.")
    print("  Chart  -> betel_palm_latlon_report.png")
    print("  Models -> models/")
    print("=" * 56)

    return results


if __name__ == "__main__":
    main()


  BETEL PALM ADVISOR — LAT/LON ML PIPELINE

[1/4] Generating geo-aware synthetic training dataset ...
      Dataset shape   : (3000, 13)
      Lat range       : -19.8 to 28.0
      Lon range       : -77.9 to 178.0
      Season classes  : {0: np.int64(1372), 3: np.int64(741), 2: np.int64(609), 1: np.int64(278)}
      Drought classes : {0: np.int64(2335), 1: np.int64(356), 2: np.int64(309)}

[2/4] Training models with lat/lon features ...

────────────────────────────────────────────────────
  Model : season_label
  Test accuracy : 95.5%
  CV accuracy   : 95.1% +/- 0.7%

                               precision    recall  f1-score   support

      Early monsoon (May-Aug)       0.95      0.97      0.96       274
     Summer monsoon (Jun-Sep)       1.00      0.86      0.92        56
       Post-monsoon (Jul-Oct)       0.97      0.98      0.97       122
Pre-monsoon coastal (Apr-Jul)       0.94      0.95      0.95       148

                     accuracy                           0.95       

In [ ]:
# predict_location.py
# Run this AFTER betel_palm_ml.py has been run at least once
# (so the models/ folder exists with .joblib files)

import joblib
import numpy as np

# ── Load the saved trained models from disk ──────────────────────
# These were saved by save_models() when you ran betel_palm_ml.py
season_model  = joblib.load("models/season_label_rf.joblib")
drought_model = joblib.load("models/drought_label_rf.joblib")
crop_model    = joblib.load("models/crop_label_rf.joblib")

# ── Encoding maps (must match betel_palm_ml.py exactly) ──────────
ZONE_MAP    = {"tropical": 0, "subtropical": 1, "semiarid": 2, "coastal": 3}
SOIL_MAP    = {"loamy": 0, "clay": 1, "sandy": 2, "laterite": 3, "alluvial": 4}
IRR_MAP     = {"yes": 0, "partial": 1, "no": 2}
REGION_MAP  = {"india_south": 0, "india_ne": 1, "sea": 2, "pacific": 3, "other": 4}

SEASON_LABELS  = ["Early monsoon (May-Aug)", "Summer monsoon (Jun-Sep)",
                   "Post-monsoon (Jul-Oct)", "Pre-monsoon coastal (Apr-Jul)"]
DROUGHT_LABELS = ["Low risk", "Medium risk", "High risk"]
CROP_LABELS    = ["Banana", "Coconut", "Tapioca/Cassava",
                   "Black pepper", "Turmeric", "Jackfruit"]

# ── Your farm location: 12.01131 N, 78.40791 E ───────────────────
# Dharmapuri district, Tamil Nadu
farm = {
    "zone":         "semiarid",
    "soil":         "laterite",
    "rainfall_mm":  850,
    "temp_c":       29,
    "irrigation":   "yes",
    "region":       "india_south",
    "humidity_pct": 62,
    "elevation_m":  380,
}

# ── Encode inputs into a 1-row NumPy array ────────────────────────
x = np.array([[
    ZONE_MAP[farm["zone"]],
    SOIL_MAP[farm["soil"]],
    farm["rainfall_mm"],
    farm["temp_c"],
    IRR_MAP[farm["irrigation"]],
    REGION_MAP[farm["region"]],
    farm["humidity_pct"],
    farm["elevation_m"],
]])

# ── Run all three models ──────────────────────────────────────────
models  = [season_model,   drought_model,   crop_model]
labels  = [SEASON_LABELS,  DROUGHT_LABELS,  CROP_LABELS]
names   = ["Season",       "Drought Risk",  "Shade Crop"]

print("=" * 52)
print("  DHARMAPURI LOCATION PREDICTION")
print("  Coords: 12.01131 N, 78.40791 E")
print("=" * 52)

for model, label_list, name in zip(models, labels, names):
    pred       = model.predict(x)[0]           # top predicted class index
    proba      = model.predict_proba(x)[0]     # probability for every class
    prediction = label_list[pred]              # convert index to label
    confidence = f"{proba.max() * 100:.1f}%"  # highest probability as %

    print(f"\n  {name}")
    print(f"    Prediction  : {prediction}")
    print(f"    Confidence  : {confidence}")
    print(f"    Breakdown   :")
    for cls, p in zip(label_list, proba):
        bar = "#" * int(p * 30)   # simple ASCII bar chart
        print(f"      {cls:<35}: {p*100:5.1f}%  {bar}")

  DHARMAPURI LOCATION PREDICTION
  Coords: 12.01131 N, 78.40791 E

  Season
    Prediction  : Post-monsoon (Jul-Oct)
    Confidence  : 67.5%
    Breakdown   :
      Early monsoon (May-Aug)            :   6.3%  #
      Summer monsoon (Jun-Sep)           :  10.6%  ###
      Post-monsoon (Jul-Oct)             :  67.5%  ####################
      Pre-monsoon coastal (Apr-Jul)      :  15.6%  ####

  Drought Risk
    Prediction  : Medium risk
    Confidence  : 53.9%
    Breakdown   :
      Low risk                           :  28.1%  ########
      Medium risk                        :  53.9%  ################
      High risk                          :  17.9%  #####

  Shade Crop
    Prediction  : Tapioca/Cassava
    Confidence  : 80.1%
    Breakdown   :
      Banana                             :   1.4%  
      Coconut                            :  14.8%  ####
      Tapioca/Cassava                    :  80.1%  ########################
      Black pepper                       :   3.8%  #
     